# dsfb-forensics reproducibility notebook

This notebook rebuilds **`dsfb-forensics`** from scratch and demonstrates exactly how the crate behaves when it audits a deterministic multi-channel trace. The point of the notebook is reproducibility rather than convenience: it clones the repository, installs the Rust toolchain, creates a known input trace, compiles the crate, executes the CLI, reads the emitted JSON artifacts, and visualizes the resulting forensic evidence using Pandas and Matplotlib.

`dsfb-forensics` is the reference specification and audit layer for the DSFB stack. It wraps the existing DSFB observer with a deterministic forensic engine that reconstructs a trust-gated causal graph, detects shatter events when measured slew exceeds the deterministic envelope, detects silent failures when an EKF baseline accepts updates that DSFB rejects or materially down-weights, and writes fresh timestamped outputs for every run.

Read the crate README for the deeper explanation of the theory links, file layout, output semantics, and CLI contract.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

repo_root = Path('/content/dsfb')
if repo_root.exists():
    shutil.rmtree(repo_root)

subprocess.run(['git', 'clone', 'https://github.com/infinityabundance/dsfb.git', str(repo_root)], check=True)

cargo_bin = Path('/root/.cargo/bin')
if not cargo_bin.exists():
    subprocess.run("curl https://sh.rustup.rs -sSf | sh -s -- -y", shell=True, check=True)

os.environ['PATH'] = f"{cargo_bin}:{os.environ['PATH']}"
subprocess.run(['rustc', '--version'], check=True)
subprocess.run(['cargo', '--version'], check=True)
repo_root

In [ ]:
import pandas as pd

trace = pd.DataFrame(
    [
        (0, 0.1, 0.00, 1.00, 0.10, 0.00, 0.02, -0.01),
        (1, 0.1, 0.10, 1.01, 0.10, 0.11, 0.09, 0.10),
        (2, 0.1, 0.21, 1.02, 0.10, 0.20, 0.22, 0.21),
        (3, 0.1, 0.33, 1.03, 0.10, 0.34, 0.32, 0.33),
        (4, 0.1, 0.46, 1.04, 0.10, 0.45, 0.47, 0.46),
        (5, 0.1, 0.60, 1.05, 0.10, 0.59, 0.61, 1.20),
        (6, 0.1, 0.75, 1.06, 0.10, 0.76, 0.74, 1.85),
        (7, 0.1, 0.91, 1.07, 0.10, 0.90, 0.92, 0.30),
        (8, 0.1, 1.08, 1.08, 0.10, 1.07, 1.10, 1.09),
        (9, 0.1, 1.26, 1.09, 0.10, 1.27, 1.24, 1.28),
        (10, 0.1, 1.45, 1.10, 0.10, 1.44, 1.46, 1.45),
        (11, 0.1, 1.65, 1.11, 0.10, 1.66, 1.64, 1.65),
    ],
    columns=[
        'step', 'dt', 'truth_phi', 'truth_omega', 'truth_alpha',
        'measurement_0', 'measurement_1', 'measurement_2'
    ],
)

trace_path = repo_root / 'crates' / 'dsfb-forensics' / 'fixtures' / 'colab_trace.csv'
trace.to_csv(trace_path, index=False)
trace.head()

In [ ]:
build_cmd = [
    'cargo', 'build', '--release', '--manifest-path', 'crates/dsfb-forensics/Cargo.toml'
]
build_result = subprocess.run(
    build_cmd,
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
if build_result.stdout:
    print(build_result.stdout)
if build_result.stderr:
    print(build_result.stderr)

run_cmd = [
    'cargo', 'run', '--release', '--manifest-path', 'crates/dsfb-forensics/Cargo.toml', '--',
    '--input-trace', 'crates/dsfb-forensics/fixtures/colab_trace.csv',
    '--slew-threshold', '6.0',
    '--trust-alpha', '0.20',
    '--baseline-comparison', 'on',
    '--report-format', 'both',
]
run_result = subprocess.run(
    run_cmd,
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
if run_result.stdout:
    print(run_result.stdout)
if run_result.stderr:
    print(run_result.stderr)

output_root = repo_root / 'output-dsfb-forensics'
run_dir = max((path for path in output_root.iterdir() if path.is_dir()), key=lambda path: path.name)
notebook_artifacts_dir = run_dir / 'notebook_artifacts'
notebook_artifacts_dir.mkdir(exist_ok=True)

(notebook_artifacts_dir / 'build_command.txt').write_text(' '.join(build_cmd) + '\n', encoding='utf-8')
(notebook_artifacts_dir / 'build.log').write_text(build_result.stdout + build_result.stderr, encoding='utf-8')
(notebook_artifacts_dir / 'run_command.txt').write_text(' '.join(run_cmd) + '\n', encoding='utf-8')
(notebook_artifacts_dir / 'run.log').write_text(run_result.stdout + run_result.stderr, encoding='utf-8')
shutil.copy2(trace_path, notebook_artifacts_dir / trace_path.name)

run_dir

In [ ]:
import json
from pandas import json_normalize

with open(run_dir / 'causal_trace.json', 'r', encoding='utf-8') as handle:
    causal_trace = json.load(handle)

with open(run_dir / 'forensic_report.json', 'r', encoding='utf-8') as handle:
    report = json.load(handle)

steps_df = json_normalize(causal_trace['steps'])
updates_df = json_normalize(
    causal_trace['steps'],
    record_path='updates',
    meta=['step_index', 'silent_failures', ['graph', 'connected_components']],
)

steps_csv_path = notebook_artifacts_dir / 'steps_table.csv'
updates_csv_path = notebook_artifacts_dir / 'updates_table.csv'
summary_json_path = notebook_artifacts_dir / 'report_summary_from_notebook.json'
manifest_json_path = notebook_artifacts_dir / 'notebook_manifest.json'

steps_df.to_csv(steps_csv_path, index=False)
updates_df.to_csv(updates_csv_path, index=False)
summary_json_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
manifest_json_path.write_text(
    json.dumps(
        {
            'run_dir': str(run_dir),
            'trace_copy': str((notebook_artifacts_dir / trace_path.name).relative_to(run_dir)),
            'tables': [
                str(steps_csv_path.relative_to(run_dir)),
                str(updates_csv_path.relative_to(run_dir)),
            ],
            'logs': [
                str((notebook_artifacts_dir / 'build.log').relative_to(run_dir)),
                str((notebook_artifacts_dir / 'run.log').relative_to(run_dir)),
            ],
        },
        indent=2,
    ),
    encoding='utf-8',
)

report_series = pd.Series(report)
display(report_series)
display(steps_df[['step_index', 'rule_id', 'trust_score', 'shatter_event', 'silent_failures', 'graph.connected_components']])
updates_df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)

for channel, group in updates_df.groupby('channel'):
    axes[0].plot(group['step_index'], group['trust_score'], marker='o', label=channel)

axes[0].set_title('Forensic trust score by channel')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Trust score')
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].bar(steps_df['step_index'] - 0.15, steps_df['graph.connected_components'], width=0.3, label='Connected components')
axes[1].bar(steps_df['step_index'] + 0.15, steps_df['silent_failures'], width=0.3, label='Silent failures')
axes[1].set_title('Fragmentation and silent failures per step')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Count')
axes[1].grid(True, axis='y', alpha=0.3)
axes[1].legend()

dashboard_png_path = notebook_artifacts_dir / 'forensic_dashboard.png'
fig.savefig(dashboard_png_path, dpi=180, bbox_inches='tight')

plt.show()
print('Saved notebook figure:', dashboard_png_path)
print('Markdown report:', run_dir / 'forensic_report.md')

In [ ]:
archive_base = output_root / run_dir.name
archive_path = archive_base.with_suffix('.zip')
zip_manifest_path = notebook_artifacts_dir / 'zip_manifest.json'

zip_manifest_path.write_text(
    json.dumps(
        {
            'archive_path': str(archive_path),
            'archived_folder': run_dir.name,
            'archived_files': sorted(
                str(path.relative_to(run_dir))
                for path in run_dir.rglob('*')
                if path.is_file()
            ),
        },
        indent=2,
    ),
    encoding='utf-8',
)

archive_path = Path(
    shutil.make_archive(str(archive_base), 'zip', root_dir=output_root, base_dir=run_dir.name)
)
print(f'Created archive: {archive_path}')
print(f'Archive contains folder: {run_dir.name}/')

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('google.colab.files is not available in this environment; download the zip manually.')

archive_path